<a href="https://colab.research.google.com/github/anamraies05-cyber/Fake_news_project/blob/main/bert_fake_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers datasets scikit-learn accelerate


In [4]:
from datasets import load_dataset

data = load_dataset("ComplexDataLab/Misinfo_Datasets", "liar")
print(data)
print(data["train"][0])

README.md:   0%|          | 0.00/15.9k [00:00<?, ?B/s]

liar/liar_train.parquet: reconstructing file:   0%|          |  0.00B / 1.20MB            

liar/liar_train.parquet: downloading bytes:           |  0.00B            

liar/liar_test.parquet: reconstructing file:   0%|          |  0.00B /  327kB            

liar/liar_test.parquet: downloading bytes:           |  0.00B            

liar/liar_validation.parquet: reconstructing file:   0%|          |  0.00B /  306kB            

liar/liar_validation.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4345 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1242 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/621 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['claim', 'veracity', 'dataset', 'label', 'question', 'answer', 'positive_evidence_text', 'positive_evidence_title', 'negative_evidence_text', 'negative_evidence_title', 'split', 'source', 'source_url', 'article_content', 'article_headline', 'author', 'initial_claim', 'fact1', 'fact2', 'fact3', 'fact4', 'num_hops', 'date', 'similarity', 'like_count', 'retweet_count', 'tweet_text', 'tweet_label', 'evidence_1', 'link_evidence_1', 'source_evidence_1', 'fact_check_source', 'image', 'article_title', 'meta_data_twitter_site', 'meta_data_og_image', 'source_id', 'fact_check_url', 'language', 'tweet_id', 'subject', 'revelant_to_vaccines', 'mention_non_covid_vaccine', 'contain_misinfo_about_vaccine', 'dominant_sentiment_tweet', 'dominant_sentiment_vaccine', 'country', 'image_id', 'image_url', 'event', 'label_2', 'pos_variations', 'Q1_contain_factual_claim', 'Q2_contain_false_info', 'Q3_impact_on_public', 'Q4_harmful_to_society', 'Q5_factchecke

In [5]:
from collections import Counter

print(Counter(data["train"]["label"]))


Counter({'half-true': 853, 'false': 849, 'mostly-true': 842, 'true': 713, 'barely-true': 699, 'pants-fire': 387, 'bill-mccollum': 1, 'doug-macginnitie': 1})


In [7]:
fake_labels = ["false", "pants-fire", "barely-true"]
real_labels = ["half-true", "mostly-true", "true"]

def keep_good(example):
    return example["label"] in fake_labels + real_labels

def make_simple(example):
    return {
        "text": example["claim"],
        "label": 1 if example["label"] in fake_labels else 0,
    }

cleaned = {}
for split in ["train", "validation", "test"]:
    ds = data[split].filter(keep_good)
    ds = ds.map(make_simple, remove_columns=ds.column_names)
    cleaned[split] = ds

print({k: len(v) for k, v in cleaned.items()})
print(cleaned["train"][0])
print(Counter(cleaned["train"]["label"]))

Filter:   0%|          | 0/4345 [00:00<?, ? examples/s]

Map:   0%|          | 0/4343 [00:00<?, ? examples/s]

Filter:   0%|          | 0/621 [00:00<?, ? examples/s]

Map:   0%|          | 0/620 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1242 [00:00<?, ? examples/s]

Map:   0%|          | 0/1241 [00:00<?, ? examples/s]

{'train': 4343, 'validation': 620, 'test': 1241}
{'label': 1, 'text': 'wisconsin is on pace to double the number of layoffs this year.'}
Counter({0: 2408, 1: 1935})


In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def to_numbers(batch):
    return tokenizer(batch["text"], truncation=True,
                     padding="max_length", max_length=128)

for split in cleaned:
    cleaned[split] = cleaned[split].map(to_numbers, batched=True)

print(cleaned["train"][0]["input_ids"][:15])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4343 [00:00<?, ? examples/s]

Map:   0%|          | 0/620 [00:00<?, ? examples/s]

Map:   0%|          | 0/1241 [00:00<?, ? examples/s]

[101, 5273, 2003, 2006, 6393, 2000, 3313, 1996, 2193, 1997, 3913, 27475, 2023, 2095, 1012]


In [9]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2)

def score(pred):
    guesses = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(pred.label_ids, guesses),
        "f1": f1_score(pred.label_ids, guesses),
    }

rules = TrainingArguments(
    output_dir="results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=rules,
    train_dataset=cleaned["train"],
    eval_dataset=cleaned["validation"],
    compute_metrics=score,
)

print("Ready to train!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Ready to train!


In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.665148,0.580645,0.273743
2,0.637578,0.703669,0.611290,0.615630
3,0.637578,0.896919,0.614516,0.526733


TrainOutput(global_step=816, training_loss=0.5422850496628705, metrics={'train_runtime': 289.7329, 'train_samples_per_second': 44.969, 'train_steps_per_second': 2.816, 'total_flos': 857018485071360.0, 'train_loss': 0.5422850496628705, 'epoch': 3.0})

In [11]:
results = trainer.evaluate(cleaned["test"])
print("Final test accuracy:", round(results["eval_accuracy"], 3))
print("Final test F1 score:", round(results["eval_f1"], 3))

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.637578,0.932791,3,0.615633,0.500524


Final test accuracy: 0.616
Final test F1 score: 0.501


In [12]:
import torch

def check_news(text):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=128).to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=1)[0]
    real = round(float(probs[0]) * 100, 1)
    fake = round(float(probs[1]) * 100, 1)
    verdict = "FAKE" if fake > real else "REAL"
    print(f"{verdict}  (real: {real}%, fake: {fake}%)")
    print("-", text)

check_news("The government will give free laptops to every student in India next month.")
check_news("The unemployment rate fell to 4 percent last year.")
check_news("Drinking hot water cures all types of cancer.")

FAKE  (real: 17.3%, fake: 82.7%)
- The government will give free laptops to every student in India next month.
REAL  (real: 93.2%, fake: 6.8%)
- The unemployment rate fell to 4 percent last year.
FAKE  (real: 3.1%, fake: 96.9%)
- Drinking hot water cures all types of cancer.


In [14]:
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/fake_news_bert"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Saved to:", save_path)

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/fake_news_bert
